# VedaVision — Deep Learning Benchmark (Health Assessment)

**Companion to `VedaVision_DL_Benchmark.ipynb` (species-ID).** Same discipline, same
purpose: an honest CNN baseline for the two-stage health classifier
(Stage 1: Healthy/Unhealthy, Stage 2: Moderate/High severity), evaluated exactly like
the handcrafted pipeline so any DL-vs-handcrafted comparison is defensible in the viva.

### Two rules from the handcrafted pipeline that carry over unchanged here — non-negotiable:

1. **No enhancement before health feature extraction.** The species-ID branch applies
   bilateral filter + CLAHE + unsharp mask; the health branch never does, because those
   steps distort exactly the colour/texture signals (yellowing, browning, lesion
   contrast) health assessment depends on. This notebook loads images from
   `masked_raw/` only — never from an enhanced branch.
2. **No colour-jitter augmentation for health.** `HueSaturationValue`/`BrightnessContrast`
   and anything CoarseDropout-like are excluded from your `augmentation.py` for the
   health branch specifically because they fabricate fake lesions or false
   yellowing/browning. The augmentation cell below only uses geometric transforms
   (flip/rotate) for the same reason — a DL run that used colour jitter here would not
   be a fair or valid comparison.

### Species-conditioning note

Your handcrafted health feature vector leans heavily on species-relative z-scores
(19 of 151 features) plus a 12-way species one-hot, because "unhealthy" looks different
per species and a raw colour threshold isn't comparable across species. A CNN given only
raw pixels has no equivalent unless you also give it species as an input — so this
notebook uses a **two-input model** (image branch + species one-hot branch) to keep the
comparison fair. An image-only CNN would be handicapped relative to your handcrafted
model in a way that has nothing to do with DL vs handcrafted features per se.

Toggle `STAGE` below to run Stage 1 or Stage 2 (run Stage 1 first, then Stage 2 — Stage 2
trains only on the unhealthy subset).


In [ ]:
# ------------------------------------------------------------------
# 0. Config — EDIT THESE
# ------------------------------------------------------------------
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Manifest CSV — same convention as your handcrafted health CSVs, needs columns:
#   image_path, species, leaf_id, is_test, stage1_label, stage2_label
# stage1_label: "healthy" / "unhealthy"
# stage2_label: "moderate" / "high"  (blank/NaN for healthy rows — ignored in Stage 1,
#                                      required for Stage 2)
# image_path must point at masked_raw images (NOT enhanced).
MANIFEST_CSV = "health_manifest.csv"      # <-- EDIT

IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
OUTPUT_DIR = Path("dl_benchmark_outputs_health")
OUTPUT_DIR.mkdir(exist_ok=True)

# Reuse the leaf-domain-adapted backbone from the species-ID notebook if you trained
# Stage 0 there — gives the health CNN the same head start. Optional.
SPECIES_STAGE0_WEIGHTS = Path("dl_benchmark_outputs/stage0_backbone.weights.h5")
USE_SPECIES_ID_BACKBONE = SPECIES_STAGE0_WEIGHTS.exists()

STAGE = 1   # 1 = Healthy/Unhealthy, 2 = Moderate/High (run 1 first)


## 1. Load manifest, build leaf-grouped split

Same rule as species-ID: `leaf_id` groups top/bottom views of one physical leaf so they
never straddle train/val, and `is_test` rows are frozen aside and touched once.


In [ ]:
df = pd.read_csv(MANIFEST_CSV)
assert {"image_path", "species", "leaf_id", "is_test", "stage1_label"}.issubset(df.columns)

species_list = sorted(df.species.unique())
species_to_idx = {s: i for i, s in enumerate(species_list)}
N_SPECIES = len(species_list)

df["species_idx"] = df.species.map(species_to_idx)
df["y_stage1"] = (df.stage1_label.str.lower() == "unhealthy").astype(int)  # 0=healthy,1=unhealthy

print(df.groupby(["stage1_label", "is_test"]).size().unstack(fill_value=0))

train_pool = df[~df.is_test].reset_index(drop=True)
test_df    = df[df.is_test].reset_index(drop=True)
print(f"train pool: {len(train_pool)}  sealed test: {len(test_df)}")


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

if STAGE == 1:
    y_for_split = train_pool.y_stage1
    frame = train_pool
else:
    # Stage 2: unhealthy rows only, from BOTH train pool and sealed test (test frozen
    # separately, same as Stage 1)
    frame_all_unhealthy = df[df.stage1_label.str.lower() == "unhealthy"].copy()
    frame_all_unhealthy["y_stage2"] = (frame_all_unhealthy.stage2_label.str.lower() == "high").astype(int)
    train_pool2 = frame_all_unhealthy[~frame_all_unhealthy.is_test].reset_index(drop=True)
    test_df2    = frame_all_unhealthy[frame_all_unhealthy.is_test].reset_index(drop=True)
    frame = train_pool2
    y_for_split = train_pool2.y_stage2
    print(f"Stage 2 train pool: {len(train_pool2)}  sealed test: {len(test_df2)}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf.split(frame, y_for_split, groups=frame.leaf_id))
tr_df = frame.iloc[train_idx].reset_index(drop=True)
va_df = frame.iloc[val_idx].reset_index(drop=True)
assert set(tr_df.leaf_id) & set(va_df.leaf_id) == set()
print(f"train: {len(tr_df)}  val: {len(va_df)}")


## 2. tf.data pipelines — geometric augmentation only (no colour jitter, no dropout)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
Y_COL = "y_stage1" if STAGE == 1 else "y_stage2"

def load_image(path, species_idx, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    species_onehot = tf.one_hot(species_idx, N_SPECIES)
    return (img, species_onehot), label

def augment(inputs, label):
    img, species_onehot = inputs
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    # NOTE: deliberately no random_brightness / random_contrast / hue / saturation here —
    # see the header note on why colour jitter is excluded for the health branch.
    return (img, species_onehot), label

def make_dataset(frame, training):
    paths = frame.image_path.values
    species_idx = frame.species_idx.values
    labels = frame[Y_COL].values
    ds = tf.data.Dataset.from_tensor_slices((paths, species_idx, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(tr_df, training=True)
val_ds   = make_dataset(va_df, training=False)

eval_test_df = test_df if STAGE == 1 else test_df2
test_ds = make_dataset(eval_test_df, training=False)   # not touched until Section 5


## 3. Two-input model: image CNN branch + species one-hot branch

Backbone loads the species-ID Stage 0 leaf-domain weights if available (consistent
starting point across both benchmark notebooks); otherwise plain ImageNet.


In [ ]:
def build_backbone():
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet"
    )
    if USE_SPECIES_ID_BACKBONE:
        base.load_weights(str(SPECIES_STAGE0_WEIGHTS))
        print("Loaded leaf-domain-adapted backbone from species-ID Stage 0.")
    return base

base = build_backbone()
base.trainable = False

img_input = tf.keras.Input(shape=(224, 224, 3), name="image")
species_input = tf.keras.Input(shape=(N_SPECIES,), name="species_onehot")

x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(img_input)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)

merged = tf.keras.layers.Concatenate()([x, species_input])
merged = tf.keras.layers.Dense(64, activation="relu")(merged)
merged = tf.keras.layers.Dropout(0.3)(merged)
output = tf.keras.layers.Dense(1, activation="sigmoid")(merged)

model = tf.keras.Model([img_input, species_input], output)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Recall(name="recall")])
model.summary()


## 4. Train — class-weighted (Stage 1's healthy/unhealthy imbalance mirrors your
handcrafted-model finding: unhealthy leaves outnumber healthy ones, which is what drove
healthy recall down to ~0.56 in the handcrafted run — watch this same metric here).


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
weights = compute_class_weight("balanced", classes=classes, y=tr_df[Y_COL].values)
class_weight = dict(zip(classes, weights))
print("class_weight:", class_weight)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

# Phase A — head only
history_a = model.fit(train_ds, validation_data=val_ds, epochs=15,
                       class_weight=class_weight, callbacks=callbacks)

# Phase B — unfreeze top of backbone, fine-tune at low LR
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Recall(name="recall")])
history_b = model.fit(train_ds, validation_data=val_ds, epochs=20,
                       class_weight=class_weight, callbacks=callbacks)

model.save(OUTPUT_DIR / f"vedavision_dl_health_stage{STAGE}.keras")


## 5. Evaluate ONCE on sealed test

Report both overall metrics and the **per-class recall breakdown** — for Stage 1,
healthy recall specifically is the number your handcrafted model struggled with
(~0.556); that's the one to compare, not just overall accuracy/F1.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

y_true = eval_test_df[Y_COL].values
y_pred_probs = model.predict(test_ds).ravel()
y_pred = (y_pred_probs >= 0.5).astype(int)

label_names = (["healthy", "unhealthy"] if STAGE == 1 else ["moderate", "high"])

print(f"SEALED TEST f1_macro: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(classification_report(y_true, y_pred, target_names=label_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=label_names, yticklabels=label_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"DL Stage {STAGE} — sealed test")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"dl_health_stage{STAGE}_confusion_matrix.png", dpi=150)
plt.show()

if STAGE == 1:
    healthy_recall = cm[0, 0] / cm[0].sum()
    print(f"Healthy recall (compare to handcrafted ~0.556): {healthy_recall:.3f}")


## 6. McNemar's test vs the handcrafted health model

Export your handcrafted Stage-N sealed-test predictions to
`handcrafted_health_stageN_predictions.csv` with columns `image_path,y_true,y_pred`
(same 60-ish sealed-test images, same order as `eval_test_df.image_path`), then compare.


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

hc_path = f"handcrafted_health_stage{STAGE}_predictions.csv"
handcrafted_preds = pd.read_csv(hc_path).set_index("image_path").loc[eval_test_df.image_path].reset_index()

dl_correct = (y_pred == y_true)
hc_correct = (handcrafted_preds.y_pred.values == handcrafted_preds.y_true.values)

both_correct = np.sum(dl_correct & hc_correct)
dl_only      = np.sum(dl_correct & ~hc_correct)
hc_only      = np.sum(~dl_correct & hc_correct)
both_wrong   = np.sum(~dl_correct & ~hc_correct)

table = [[both_correct, dl_only], [hc_only, both_wrong]]
result = mcnemar(table, exact=True)
print(table)
print(f"McNemar p-value: {result.pvalue:.4f}")


## 7. Grad-CAM — is the CNN looking at lesions, or something else?

For health specifically this is the most important interpretability check: does the
model's attention land on holes/spots/discolouration (what your handcrafted `holes.py`,
`spots.py`, `colour_health.py` explicitly target), or does it spread over the whole leaf
/ background edge — which would suggest it's picking up a diffuse texture or colour-cast
shortcut rather than the actual lesion evidence a plant pathologist would look for.


In [ ]:
def make_gradcam_heatmap(img_array, species_onehot_array, model, base_model, last_conv_layer_name="Conv_1"):
    grad_model = tf.keras.models.Model(
        [model.inputs], [base_model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model([img_array, species_onehot_array])
        class_channel = predictions[:, 0]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

example_row = eval_test_df.iloc[0]
img = tf.io.read_file(example_row.image_path)
img = tf.image.decode_jpeg(img, channels=3)
img = tf.image.resize(img, IMG_SIZE)
img_array = tf.expand_dims(img, 0)
species_arr = tf.expand_dims(tf.one_hot(example_row.species_idx, N_SPECIES), 0)

heatmap = make_gradcam_heatmap(img_array, species_arr, model, base)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(img.numpy().astype("uint8")); plt.title("Original"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(img.numpy().astype("uint8")); plt.imshow(heatmap, cmap="jet", alpha=0.5)
plt.title("Grad-CAM"); plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"gradcam_health_stage{STAGE}_example.png", dpi=150)
plt.show()


## 8. Running Stage 2

Set `STAGE = 2` in the config cell and re-run the notebook top-to-bottom. Stage 2 trains
and evaluates only on unhealthy leaves (Moderate vs High) — the Low/Mid collapse you
already established for the handcrafted model applies here too: don't try to recover a
3-class DL Stage 2, the boundary problem is a labeling/data issue, not something a
different model architecture fixes.

## 9. What to put in the dissertation from this notebook

- Stage 1 sealed-test f1_macro **and healthy recall specifically**, next to the
  handcrafted SVM's 0.885 f1 / recall figures.
- Stage 2 sealed-test f1_macro on Moderate/High, next to the handcrafted result.
- McNemar p-values for both stages.
- Grad-CAM examples — ideally one clean case and one of your known-hard cases (e.g. a
  leaf-miner-trail image, since that was a confirmed handcrafted under-detection issue —
  worth seeing whether the CNN catches what your handcrafted `miner_trail.py` missed).
